# Development and validation of a machine learning model for metabolic syndrome risk prediction using routinely available clinical indicators: a CHARLS cohort study with external validation Exploration with `mlcroissant`
This notebook provides an end-to-end example for loading and exploring a dataset described by a Croissant schema using the `mlcroissant` library.

### Dataset Source
This dataset is described by the following Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.53cj-dkrc/fair2.json`


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and data records from the Croissant dataset using `mlcroissant`. This allows easy exploration and processing of structured scientific data.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant metadata URL
croissant_url = "https://sen.science/doi/10.71728/senscience.53cj-dkrc/fair2.json"

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Title: {metadata.name}\n")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
Let's explore the available record sets (tables/CSV files, etc.), field definitions, and how to reference them using their Croissant `@id` fields.

**Each entity (RecordSet, Field, Column) in Croissant is uniquely referenced by its `@id`.**

Let's list all available record sets and their field definitions by `@id`.

In [ ]:
# Explore available record sets
print("Record sets available in this dataset:")
record_sets_info = []
for rs in metadata.record_sets:
    print(f"- RecordSet name: {rs.name} | @id: {rs.id}")
    record_sets_info.append({
        "name": rs.name,
        "id": rs.id,
        "fields": rs.fields
    })
    # Show basic info about fields
    print("    Fields (by @id):")
    for field in rs.fields:
        print(f"        - {field.name} (@id: {field.id}, dataType: {field.data_type})")
    print()

# If no record sets, fail gracefully
if len(record_sets_info) == 0:
    print("No record sets found in this dataset. Please check the Croissant schema.")

## 3. Data Extraction
Load data from each record set into a pandas DataFrame for analysis, referencing using their Croissant `@id`s.

If there is more than one record set, we'll load them all into a dictionary. When referencing fields, always use their `@id`.

In [ ]:
dataframes = {}
print("\nLoading all record sets into pandas DataFrames:")
for rs in metadata.record_sets:
    print(f"Loading RecordSet: {rs.name} (@id: {rs.id}) ...")
    records = list(dataset.records(record_set=rs.id))
    df = pd.DataFrame(records)
    dataframes[rs.id] = df
    print(f"    Loaded {len(df)} rows, columns: {df.columns.tolist()}\n")

# For demonstration, pick the first record set as the main dataset
main_record_set_id = metadata.record_sets[0].id if metadata.record_sets else None
if main_record_set_id is not None:
    print(f"Columns in DataFrame for RecordSet @id={main_record_set_id}:\n", dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Let's do some basic data analysis: filtering, normalization, grouping, etc. **All field (column) references are by their Croissant `@id`.**

Choose a numeric field and a grouping field (e.g., 'age', 'gender', diagnosis column, etc.) by their `@id`, and perform common analysis steps.

> **TIP:** Check the output of the previous cell for available columns (`@id`).

In [ ]:
# Example: Choose field IDs from field definitions above.
# If unsure, print columns
df = dataframes[main_record_set_id]
print(f"Columns available: {df.columns.tolist()}")

# Suggest fields by choosing ones with float/int types (edit as needed after inspecting actual field IDs):
numeric_field_id = None
group_field_id = None
# Try to auto-detect a numeric field and a group field
for rs in metadata.record_sets:
    if rs.id == main_record_set_id:
        for f in rs.fields:
            # Heuristically pick first numeric (float/int) and first categorical/text
            if numeric_field_id is None and f.data_type in ['schema:Number', 'schema:Float', 'schema:Integer']:
                numeric_field_id = f.id
            if group_field_id is None and f.data_type in ['schema:Text', 'schema:Boolean']:
                group_field_id = f.id
        break

print(f"Chosen numeric field: {numeric_field_id}")
print(f"Chosen group field: {group_field_id}")

# Only continue if appropriate fields are found
if numeric_field_id is None or group_field_id is None:
    print("Suitable numeric and/or group field not found. Please check field @id's and pick manually.")
else:
    # Remove missing values if any
    numeric_vals = pd.to_numeric(df[numeric_field_id], errors='coerce')

    # Example threshold: the mean value (could pick e.g. median or fixed threshold)
    threshold = numeric_vals.mean()
    filtered_df = df[numeric_field_id][numeric_vals > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (using mean as threshold):\n", df.loc[numeric_vals > threshold].head())

    filtered_df = df.loc[numeric_vals > threshold].copy()
    filtered_df[f"{numeric_field_id}_normalized"] = (numeric_vals[numeric_vals > threshold] - numeric_vals.mean()) / numeric_vals.std()

    print(f"\nNormalized values for {numeric_field_id}:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by another field if it is present
    if group_field_id in df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"\nGrouped by {group_field_id} (means):")
        print(grouped_df.head())

## 5. Visualization
Visualize the distribution of the chosen numeric field and/or explore group-wise differences using common plots.

> You'll need `matplotlib` and/or `seaborn` (install with `!pip install matplotlib seaborn` if needed).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution for the chosen numeric field
if numeric_field_id is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(pd.to_numeric(df[numeric_field_id], errors='coerce').dropna(), kde=True)
    plt.title(f"Distribution of '{numeric_field_id}'")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

# Boxplot of numeric by group
if numeric_field_id is not None and group_field_id is not None:
    plt.figure(figsize=(10,5))
    sns.boxplot(x=df[group_field_id], y=pd.to_numeric(df[numeric_field_id], errors='coerce'))
    plt.title(f"Boxplot of '{numeric_field_id}' grouped by '{group_field_id}'")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion
In this notebook you:
- Loaded the dataset, complete with rich metadata, using Croissant and the `mlcroissant` Python library
- Explored available record sets, fields, and referenced all fields/columns via their Croissant `@id`
- Loaded one or more record sets into pandas and performed exploratory data analysis (EDA)
- Visualized the distributions and group differences for key features by referencing their `@id`

This approach ensures reproducibility and portability for data science workflows across FAIR data packages using MLCommons Croissant.